# Etapa 3 — Contratos e dados sintéticos

Gera seis fontes fictícias em três lotes de chegada, no Volume de entrada. Mantém dados inválidos intencionais para testar a futura quarentena. Esta etapa valida arquivos e persistência; as regras de negócio serão aplicadas na Prata.

Execute somente o primeiro CMD, preencha os widgets, depois execute os CMDs seguintes. Não é necessário instalar bibliotecas. Se o pacote medallion já estiver importado nesta sessão, reinicie o Python antes de começar para carregar os novos módulos.


In [ ]:
# CMD 1 — Exibir os parâmetros, sem iniciar a geração.
# catalogo: workspace. Usa o catálogo já validado.
# data_referencia: 2026-01-15. Data BASE dos eventos sintéticos; não é a data de hoje.
# Mantenha essa data nesta rodada para reproduzir os cenários e resultados esperados.
# batch_id: deixe vazio. Gera um UUID para identificar esta execução auditada.
# Preencha batch_id apenas se precisar associar tentativas ao mesmo lote lógico.
# Não há raiz_arquivos: usamos o Volume padrão já planejado no notebook 01.
dbutils.widgets.text("catalogo", spark.catalog.currentCatalog())
dbutils.widgets.text("data_referencia", "2026-01-15")
dbutils.widgets.text("batch_id", "")
print("Preencha catalogo=workspace, data_referencia=2026-01-15 e deixe batch_id vazio. Depois execute os próximos CMDs.")


In [ ]:
# CMD 2 — Localizar o repositório e importar os módulos. Nenhum preenchimento adicional.
from pathlib import Path
import sys
atual = Path.cwd().resolve()
raiz = next((p for p in [atual, *atual.parents]
             if (p / "src" / "medallion").is_dir() and (p / "pyproject.toml").is_file()), None)
if raiz is None:
    raise RuntimeError("Abra este notebook dentro da pasta Git do projeto.")
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
from medallion.config import PipelineConfig
from medallion.synthetic.execute import executar_geracao


In [ ]:
# CMD 3 — Ler os widgets e validar antes de gravar. Conferir os valores no output.
parametros = {"ambiente": "dev", "modo": "incremental",
              "catalogo": dbutils.widgets.get("catalogo").strip(),
              "data_referencia": dbutils.widgets.get("data_referencia").strip()}
batch_id = dbutils.widgets.get("batch_id").strip()
if batch_id:
    parametros["batch_id"] = batch_id
config = PipelineConfig(**parametros)
print("Catálogo:", config.catalogo)
print("Data base sintética:", config.data_referencia)
print("Batch desta execução:", config.batch_id)
print("Volume de entrada:", config.caminho_entrada)


In [ ]:
# CMD 4 — Criar Volume de entrada, publicar e reler arquivos; testar repetição sem novas gravações.
# Esperado: DADOS SINTÉTICOS: OK, seis fontes, três lotes e auditoria SUCESSO com hashes válidos.
# A falha deliberada pertence ao notebook 02; aqui não esperamos status FALHOU.
# Se houver divergência em arquivo existente, preserve o erro e envie para revisão; não apague a pasta.
resultado = executar_geracao(spark, config, raiz)


## Coleta desta etapa

Confira `DADOS SINTÉTICOS: OK`, os três resumos de lotes e o registro final `SUCESSO` com `hashes_verificados: True`. Guarde `RUN_ID_GERACAO`, caminho do cenário e hash do manifesto. Os arquivos de dados e o manifesto ficam no Volume de entrada; a auditoria fica no Volume de evidências e tabela de controle.

Exporte como **IPYNB com outputs**, confirme que os resultados estão no arquivo e faça commit/push na branch `chore/estrutura-inicial` com a mensagem `docs: valida geracao de dados sinteticos`. Envie o export e o SHA para revisão. Uma repetição reutiliza os arquivos iguais; divergências interrompem a execução e nunca são sobrescritas. Execute este cenário com um único escritor por vez.

Os três lotes são preparados agora. A futura Bronze deverá consumi-los na ordem `01_inicial`, `02_incremental`, `03_atrasado`, em execuções separadas, para demonstrar incrementalidade e atraso. Pare nesta etapa até a revisão dos outputs.
